Before reading this notebook, make sure you have already gone through the **[Transformer Architecture and Self Attention Notebook]**, as they provide the foundational concepts needed to understand the details discussed here..

Here is the architecture of transformer, which we already discussed in previous notebook. Here we will learn about  Positional Encoding, Multihead attention, Feed Forward Layer, Add and Norm etc all components in transformer in details.

<div align="center">

[![attent.png](https://i.postimg.cc/Bb7Sm55V/attent.png)](https://postimg.cc/D4s3ZXMr)

*Source: [ Attention is all you need original paper](https://arxiv.org/abs/1706.03762)*
</div>


Let's understand first two foundational concepts: **embeddings and positional embeddings.**

When we feed text into a Transformer, the raw words or tokens cannot be directly processed by the model. Instead, each token is converted into a high-dimensional vector representation called an **embedding**. These embeddings capture semantic meaning—so that words with similar meanings (like king and queen) end up close to each other in this vector space.

However, **embeddings** alone do not contain information about the order of words in a sentence. For example, the model would treat *“the cat chased the dog” the same as “the dog chased the cat.”* To overcome this, the Transformer introduces positional embeddings (or positional encodings). These are vectors added to the token embeddings to inject information about the position of each word in the sequence.

Together, t**oken embeddings + positional embeddings** form the input representation that the Transformer encoder and decoder layers process. This combination allows the model to not only understand the meaning of words but also their relative positions—enabling it to capture both semantics and structure in the data.

## Word Embeddings
Before text can be processed by a Transformer, each token (word or subword) must be converted into a numerical representation.  
This is done using **word embeddings** — dense vectors that capture semantic meaning.

- Suppose our vocabulary size is \(V\) and the embedding dimension is $(d_{model}$).
- We create an **embedding matrix** $(E \in \mathbb{R}^{V \times d_{model}}$).
- Each token (represented by an index in the vocabulary) is mapped to a vector of size $(d_{model}$).

For example:
$
x_{token} = E[w]
$
where \(w\) is the token index and $(x_{token}$) is its embedding.

 This means that words with similar meanings (like *king* and *queen*) will have embeddings that are close in vector space.



## Why Do We Need Positional Information?
Unlike RNNs or LSTMs, Transformers do not process tokens sequentially.  
Self-attention treats the input as a **set of tokens** with no inherent order.  

But language depends on **order**:
- *"The dog chased the cat"* is different from *"The cat chased the dog"*.  

Therefore, we need a way to encode **position** (word order) into the embeddings.



## Positional Embeddings
To inject sequence order, we add a **positional embedding** (vector) to each token embedding.

Final input representation for each token:
$
z_i = x_i + p_i
$

where:
- $(x_i$) = word embedding of token $(i$)  
-$(p_i$) = positional embedding for position $(i$)


## Sinusoidal Positional Encoding (Original Transformer)
The original Transformer used **sinusoidal functions** to generate positional embeddings.  
These encodings ensure that every position has a unique representation, and nearby positions have smoothly varying values.

For position $(pos$) and dimension $(i$):

$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{\frac{2i}{d_{model}}}}\right)
$

$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{\frac{2i}{d_{model}}}}\right)
$

- Even dimensions → sine functions  
- Odd dimensions → cosine functions  


This creates wave-like patterns, where each dimension varies at a different frequency. The combination uniquely represents both **absolute position** and **relative distance** between words.  

For example, in the figure, different colors show sine and cosine waves at various frequencies, and the boxed values illustrate how words at different positions (p0, p1, p2, p3) get distinct positional vectors.  

![Positional Encoding](https://erdem.pl/static/546f1f587232f70269c5af8d617adad9/encoding.png)

**Source:** [Understanding Positional Encoding in Transformers – erdem.pl](https://erdem.pl/2021/05/understanding-positional-encoding-in-transformers/)


<div align='center'>

<img src="https://i.postimg.cc/4y501DFX/image.png" >
 <figcaption> Figure 2: Input/Output Embeddings </figcation>

</div>

It should be noted that the four boxes represent first four elements of a 512 dimensional vectors. Thus the values in the boxes in positional embedding represent the first four elements of a 512 dimensional vector. The addition of word embedding and positional embedding is element wise addition.

## Learned Positional Embeddings (Modern Variants)

Many modern models (like **BERT** and **GPT**) do not rely on sinusoidal positional encodings.  
Instead, they use **learned positional embeddings**:  

- Each position in the sequence has its own **trainable vector**.  
- These vectors are updated during training, just like word embeddings.  
- This gives the model **flexibility** to learn the best way to represent positional information for the task.  

Word Embedding (WE): $[ w1, w2, w3, w4, w5 ]$

Positional Embedding (PE): $[ p1, p2, p3, p4, p5 ]$

Final Input to Model: $[ w1+p1, w2+p2, w3+p3, w4+p4, w5+p5 ]$

Below is a simple way to imagine learned positional embeddings:


Here:  
- **WE** → Embedding of the actual word (like "cat", "sat").  
- **PE** → Learned vector for the word's position (1st, 2nd, 3rd...).  
- The sum **(WE + PE)** becomes the **final input** to the Transformer.  



```plaintext
Position Index:    1      2      3      4      5
Word Embedding:   cat    sat     on    the    mat
Positional Emb:   p1     p2     p3     p4     p5
                 -------------------------------
Final Vector:   cat+p1 sat+p2 on+p3 the+p4 mat+p5
```

*This way, BERT and GPT learn positional information directly instead of relying on fixed sine/cosine waves.*



## Putting It All Together
For each token in the input:
1. Convert the token into a **word embedding**.  
2. Add its corresponding **positional embedding**.  
3. The result is passed into the **encoder** as the input sequence.

$
z_i = E[w_i] + P[pos_i]
$

This way, the Transformer knows **what the word is** (via embeddings) and **where it is** (via positional encoding). In summarize form  Word embeddings capture **meaning** of tokens. Positional embeddings capture **order** of tokens. Together, they allow the Transformer to process sequences effectively, even without recurrence.



# Multi-Head Attention

When researchers first started working with **self-attention**, the idea was simple: instead of processing sequences step by step like RNNs, why not let each word directly **look at all the other words** and decide which ones are important? This worked really well — for example, in the sentence *“The cat sat on the mat”*, the word *“cat”* can immediately connect to *“the”* (its determiner) and *“sat”* (its verb) without having to wait for sequential processing.  

But here’s the catch: a **single self-attention mechanism** is like using only **one lens** to look at the sentence. Imagine reading a story but only being able to focus on *grammar* while ignoring *meaning*, or only seeing *short-range dependencies* while missing *long-range ones*. One head of attention tends to specialize in **just one type of relationship** — for example, which word is the subject, or which word completes the object phrase. This limitation is where the idea of **multi-head attention** was born.  

**Multi-head attention** is essentially the idea of giving the model **multiple lenses** to look at the same sentence, each with a slightly different perspective. Instead of computing just one set of queries, keys, and values ($(Q, K, V$)), the model computes several sets — one for each “head.” Each head has its own learned projection matrices ($(W^Q_i, W^K_i, W^V_i$)), so it transforms the embeddings differently. Then, each head performs self-attention separately, capturing **different aspects of the relationships** between words.  

For example, in *“The cat sat on the mat”*:  
- One head may focus on **syntactic structure** (e.g., linking *“The”* to *“cat”*).  
- Another head may focus on **long-distance dependencies** (e.g., connecting *“sat”* with *“mat”*).  
- Another may capture **semantic similarity** (e.g., relating *“cat”* to other animals in context if the sentence was longer).  

After all the heads have computed their own attention outputs, the model **concatenates their results** and projects them back into the original dimension. The final representation of each word is thus a **rich blend of multiple perspectives**, instead of being restricted to a single viewpoint.  

This mechanism is crucial for the success of Transformers because natural language has **multi-faceted dependencies**. Words are related by grammar, meaning, position, and sometimes even sound or style. By allowing the model to look through **multiple attention heads**, we let it capture these different relationships in parallel, leading to much stronger contextual representations. In fact, this is one of the main reasons Transformers replaced RNNs and became the backbone of models like BERT, GPT, and modern large language models.  



 In short: **multi-head attention** came from the need to overcome the narrow perspective of single self-attention. It works by running multiple attention mechanisms in parallel, each specializing in a different type of relationship, and then merging their knowledge. This makes the model far more powerful at understanding complex language.




[![multihead.png](https://i.postimg.cc/LX9SPS0S/multihead.png)](https://postimg.cc/wtGnKCp4)  

<div align='center'>

Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
</div>


The image shows how **multi-head self-attention** works inside the Transformer. It is a combination of **scaled dot-product attention** applied in parallel across multiple heads.


## Step 1: Input Projections
We start with the input matrix \(X\) (tokens embedded).  
From this, we create three different projections using learned matrices:

$
Q = X W_Q, \quad K = X W_K, \quad V = X W_V
$

- $(W_Q, W_K, W_V$) are separate linear transformations.  
- Shape: $(Q, K, V \in \mathbb{R}^{n \times d_k}$)



##  Step 2: Scaled Dot-Product Attention (per head)
Each head performs attention independently:

$
\text{Attention}(Q, K, V)
= \text{Softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$

- $(QK^T$): similarity between queries and keys.  
- Divide by $(\sqrt{d_k}$): scaling for numerical stability.  
- Softmax: turns scores into probabilities.  
- Multiply with $(V$): weighted sum of values.

Optional: **Masking** is applied in decoder self-attention to prevent looking ahead at future tokens.



##  Step 3: Multiple Heads
Instead of one head, we compute multiple in parallel:

$
head_i = \text{Attention}(Q W_Q^i, K W_K^i, V W_V^i)
$

Each head has its own set of weight matrices.  
This allows the model to capture **different types of relationships** simultaneously.  



## Step 4: Concatenation
The outputs of all heads are concatenated:

$
H = [head_1 \, || \, head_2 \, || \, \dots \, || \, head_h]
$

Shape: $(H \in \mathbb{R}^{n \times (h \cdot d_v)}$)



##  Step 5: Final Linear Transformation
A final projection maps concatenated heads back to the model dimension:

$
Z = H W_O
$

- $(W_O \in \mathbb{R}^{(h \cdot d_v) \times d_{model}}$)  
- Final shape: $(Z \in \mathbb{R}^{n \times d_{model}}$)



## Intuition
- Each head focuses on **different aspects** of the input.  
  - One head might align subjects with verbs.  
  - Another head might track positional or syntactic relations.  
- By combining them, the model builds a **richer representation** of the sequence.



##  Summary of Flow
1. **Linear**: Input → Q, K, V projections.  
2. **Matmul + Scale + Softmax**: Compute attention weights.  
3. **Matmul**: Apply weights to values → context vectors.  
4. **Concat**: Join outputs from multiple heads.  
5. **Linear**: Project back to model dimension.  

 This mechanism is the **core building block** of the Transformer encoder and decoder.


# Multi-Head Self-Attention Example  

We take the sentence:  
**"The cat sat on the mat"** (6 tokens).  

Each token is embedded into a vector of dimension \(d_{model} = 8\) (toy size).  



## 1. Input Embedding Matrix
Suppose the embeddings are stacked:

$
X =
\begin{bmatrix}
x_{The} \\
x_{cat} \\
x_{sat} \\
x_{on} \\
x_{the} \\
x_{mat}
\end{bmatrix}
\quad
\in \mathbb{R}^{6 \times 8}
$


## 2. Linear Projections for Each Head
For each head \(h\), we learn three projection matrices:

$
W_Q^h \in \mathbb{R}^{8 \times d_k}, \quad
W_K^h \in \mathbb{R}^{8 \times d_k}, \quad
W_V^h \in \mathbb{R}^{8 \times d_v}
$

Let’s say we use **2 heads** ($(h=1,2$)), each with $(d_k = d_v = 4$).  

So:  

$
Q^h = X W_Q^h, \quad K^h = X W_K^h, \quad V^h = X W_V^h
$

Shape: $(Q^h, K^h, V^h \in \mathbb{R}^{6 \times 4}$)  



## 3. Scaled Dot-Product Attention (per head)
For each head:

$
\text{Attention}(Q^h, K^h, V^h)
= \text{Softmax}\left(\frac{Q^h {K^h}^T}{\sqrt{d_k}}\right)V^h
$

- Compute **scores**: $(Q^h {K^h}^T \in \mathbb{R}^{6 \times 6}$)  
- Apply softmax → attention weights $(\in \mathbb{R}^{6 \times 6}$)  
- Multiply by $(V^h$) → context $(\in \mathbb{R}^{6 \times 4}$)  



## 4. Multi-Head Concatenation
Each head produces its own context:

$
head_1 = \text{Attention}(Q^1, K^1, V^1) \quad (6 \times 4)
$  

$
head_2 = \text{Attention}(Q^2, K^2, V^2) \quad (6 \times 4)
$

Concatenate:

$
H = [head_1 \, || \, head_2] \quad \in \mathbb{R}^{6 \times 8}
$



## 5. Final Projection
Apply output projection \(W_O \in \mathbb{R}^{8 \times 8}\):  

$
Z = H W_O
\quad \in \mathbb{R}^{6 \times 8}
$



## Intuition with Sentence
- Word **"sat"** can attend strongly to **"cat"** (subject) in one head.  
- Another head may capture relation between **"sat"** and **"mat"** (object).  
- Concatenating heads gives a **richer representation** of dependencies.  



## Summary of Shapes

| Step | Shape |
|------|-------|
| Input $(X$) | $(6 \times 8$) |
| Q, K, V (per head) | $(6 \times 4$) |
| Attention weights | $(6 \times 6$) |
| Output per head | $(6 \times 4$) |
| Concatenated H | $(6 \times 8$) |
| Final Z | (6 \times 8) |


 Thus, **multi-head self-attention** allows each word in *"The cat sat on the mat"* to look at different words **in multiple ways** simultaneously.  


## **Final Linear and Softmax Layer in Transformer Decoder**

In the Transformer decoder, after all **decoder layers**, we need to generate **predictions over the vocabulary**. This is done using a **linear layer followed by a softmax**. Let's break it down.


### **1. Decoder Output (After Add & Norm)**

Each decoder layer has:

- **Masked Multi-Head Attention**
- **Encoder-Decoder Attention**
- **Feed Forward Network (FFN)**

Each of these sub-layers is followed by **Add & Norm**:

$
\text{Layer Output} = \text{LayerNorm}(x + \text{Sublayer}(x))
$

After the **final decoder layer**, we get a tensor:

$
\mathbf{D} \in \mathbb{R}^{\text{seq\_len} \times d_\text{model}}
$

where:

- `seq_len` = length of target sequence
- `d_model` = hidden dimension

This is the **contextualized representation** of each token in the target sequence.



### **2. Linear Layer**

The decoder output is projected to the **vocabulary size** using a **linear layer**:

$
\mathbf{z} = \mathbf{D} \mathbf{W} + \mathbf{b}
$

Where:

- $( \mathbf{D} $) = decoder output $((\text{seq\_len} \times d_\text{model})$)  
- $( \mathbf{W} \in \mathbb{R}^{d_\text{model} \times V} $) is the weight matrix  
- $( \mathbf{b} \in \mathbb{R}^{V} $) is the bias  
- $( V $) = vocabulary size  

This gives **logits** for each token position over the vocabulary:


$
\mathbf{z} \in \mathbb{R}^{\text{seq\_len} \times V}
$



### **3. Softmax Layer**

To convert logits into probabilities:

$
P(y_t = v | y_{<t}, X) = \text{softmax}(\mathbf{z}_t) = \frac{e^{z_{t,v}}}{\sum_{v'=1}^{V} e^{z_{t,v'}}}
$

- Each row $( \mathbf{z}_t $) corresponds to the scores for the `t`-th token.  
- Softmax normalizes these scores to **probabilities over the vocabulary**.  
- The token with the **highest probability** is typically chosen as the predicted token (during inference) or used for loss computation (during training).



### **4. Summary Flow**

1. **Final decoder layer output** → shape `(seq_len, d_model)`  
2. **Linear layer** → shape `(seq_len, vocab_size)` (logits for each token)  
3. **Softmax** → shape `(seq_len, vocab_size)` (probabilities for each token)  

This forms the **final step of generating predictions** in the Transformer.  

$
\text{Output Probabilities} = \text{Softmax}(\text{Linear}(\text{Add & Norm}(\text{Decoder Output})))
$

This final step allows the Transformer decoder to **predict the next token** in the sequence based on all previous tokens and encoder information.

# ***Resources***

https://erdem.pl/2021/05/understanding-positional-encoding-in-transformers